In [1]:
import requests
import os
import numpy as np
from netCDF4 import Dataset
import datetime as dt
import shutil
import pandas as pd

In [2]:
def change_val_1D_char(ds,varname,new_val):
    var=ds.variables[varname]
    old_val=var[:].tobytes().decode().strip()
    
    print("Modifying ",varname," from ", old_val," to ",new_val)

    new_array=np.chararray(var.shape)
    new_array[:]=" " 
    for i in range(len(new_val)):
        new_array[i]=new_val[i]

    var[:]=new_array

In [3]:
def change_val_2D_char(ds,varname,dim_level,new_val):
    var=ds.variables[varname]
    old_val=var[dim_level,:].tobytes().decode().strip()
    print("Modifying ",varname," for dim_level ", dim_level, "from ", old_val," to ",new_val)

    new_type_array=np.chararray(var[dim_level,:].shape)
    new_type_array[:]=" " 
    for i in range(len(new_val)):
        new_type_array[i]=new_val[i]
    var[dim_level,:]=new_type_array[:]

In [4]:
def add_new_var(ds,varname):
    if varname == 'PROGRAM_NAME':
        new_var=ds.createVariable(varname, 'S1', ('STRING64'), fill_value=' ')
        new_var.long_name = "Name of the program"
    if varname == 'SENSOR_FIRMWARE_VERSION':
        new_var=ds.createVariable(varname, 'S1', ('N_SENSOR', 'STRING32'), fill_value=' ')
        new_var.long_name = "Firmware version of the sensor"
    if varname == 'PROJECT_NAME':
        new_var=ds.createVariable(varname, 'S1', ('STRING64'), fill_value=' ')
        new_var.long_name = "Name of the project"
    

In [5]:
def change_var_long_name(ds,varname,new_long_name):
    var=ds[varname]
    var.long_name=new_long_name

In [6]:
def change_variable_dimension_length_1D(OUT_FILE,ds_src,varname_new_length,new_length):
    #print("tutu")
    #print("DBG: changing " + varname_new_length + " length to ",new_length)

    new_dimlength='STRING'+str(new_length)
    
    new_ds = Dataset(OUT_FILE, 'w',format='NETCDF3_CLASSIC')

    #copy-paste dimension
    for dimname, dim in ds_src.dimensions.items():
        if dimname == 'N_MISSIONS':
            new_ds.createDimension(dimname, None)
        else:
            new_ds.createDimension(dimname, len(dim))

    #copy-paste variables and their local attribute:
    for varname, var in ds_src.variables.items():
        #print("\n Treating ", varname)
        #print(var.dimensions)

        #copy-paste variable apart for the new length
        if varname in [varname_new_length]:
            new_var = new_ds.createVariable(varname, var.dtype, (new_dimlength), fill_value=ds_src.variables[varname]._FillValue)
        else:
            new_var = new_ds.createVariable(varname, var.dtype, var.dimensions, fill_value=ds_src.variables[varname]._FillValue)

        #copy-paste local attributes
        for local_attrs in ds_src.variables[varname].ncattrs():
            if (local_attrs != '_FillValue'):
                new_var.setncattr(local_attrs,ds_src.variables[varname].getncattr(local_attrs))

        #copy old content
        #print("copying content for " + varname)
        if varname in [varname_new_length]:
            src_variable_length=np.size(ds_src.variables[varname][:])
            #print("src_variable_length = ",src_variable_length)
            n=min(src_variable_length,new_length)
            new_var[:n]=ds_src.variables[varname][:n]
        else:
            new_var[:]=ds_src.variables[varname][:]

    
        #if new_var[:].dtype == '|S1':
        #    print(varname, " :",new_var[:].tobytes().decode().strip())
        #else:
        #    print(varname, " :",new_var[:])

        #copy-paste global attributes
        new_ds.title = ds_src.title
        new_ds.institution = ds_src.institution
        new_ds.source = ds_src.source
        new_ds.history = ds_src.history
        new_ds.references = ds_src.references
        new_ds.user_manual_version = ds_src.user_manual_version
        new_ds.Conventions = ds_src.Conventions
        try:
            new_ds.decoder_version = ds_src.decoder_version
        except:
            s="1"
        new_ds.id = "https://doi.org/10.17882/42182"

    return new_ds
    

In [ ]:
#V5.0 (MARTEC) : 6900673, 6900674, 6900675 (correctly indicated as MARTEC in the meta file)
#V5.1 (KANNAD): 6900676, 6900677, 6900678, 6900679, 6900680,6900681, 6900682,6900683, 6900684, 6900686 (incorrectly indicated as MARTEC for all but for 1, which is indicated as an nke)
#V5.2 (KANNAD) : 6900631, 6900632 (incorrectly indicated as MARTEC)
#V5.5 (nke): 6900685, 6900939 et 6900964 (incorrectly indicated as MARTEC)

In [7]:
# Read input csv file from database
input_file="C:/Users/ddobler/Documents/03b_NVS_updates/Program_Name_preparation/Polarstern_batch.dsv"
df = pd.read_csv(input_file,keep_default_na=False)
#print(df)
list_wmos=df['PLATFORM_CODE'].values
list_sensor_mdl=df['SENSOR_MODEL'].values
list_sensor_fm=df['SENSOR_FIRMWARE_VERSION'].values
#list_sensor_fm[np.where(isnan(list_sensor_fm))]=''
list_prog_name=df['PROGRAM_NAME'].values
print(list_sensor_fm)

['7.2.5' '7.2.5' '7.2.5' '' '' '' '' '' '' 'V2' '' '' '' '' '' '' '' '' ''
 '' '' '' '7.2.5' '' '' '' 'V2' '' '' '' '' '' '' '' '' '' '' '' '' '' ''
 '' '' '' '' '' '' '' '' '' '' '' '' '' '' '' '' '' '' '' '7.2.5' '' '' ''
 '' '' '' '' '' '' '' '' '' '' '' '' '' '' '' '' '' '' '' '' '' '' '' ''
 '' '' '' '' '' '' '' '' '' '' '' '' '' '' '7.2.5' '' '' '' '' '' '' '' ''
 '' '' '' '' '' '' '' '' '' '' '' '' 'V3' '' '' '' '' '' '' '' '' '' '' ''
 '' '' '' '' '' '' '' '' '' '' '' 'V2' '7.2.5' '' '' '' '' '' '7.2.5' ''
 '7.2.5' '' '' '' '' '' '' '' '7.2.5' '1.3' '' '' '' '7.2.5' '7.2.5' '' ''
 '' '' '' '' '' '' '' '' '' '' '7.2.5' '' '7.2.5' '7.2.5' '7.2.5' '7.2.5'
 '7.2.5' '7.2.5' '7.2.5' '7.2.5' '' '' '7.2.5' '' '' '' '' '' '' '' 'V3'
 'V3' '' 'V3' 'V3' 'V3' 'V3' 'V3' 'V3' 'V3' 'V3' 'V3' 'V3' 'V3' 'V3' 'V3'
 'V3' 'V3' 'V3' 'V3' '' 'V3' '' '' '' '' '' 'V3' 'V3' 'V3' 'V3' 'V3' 'V3'
 'V3' 'V3' 'V3' 'V3' 'V3' 'V3' 'V3' 'V3' 'V3' 'V3' 'V3' 'V3' 'V3' 'V3'
 'V3' 'V3' 'V3' 'V3' 'V3' 'V3' 'V3' '' 

In [ ]:
#list_wmos=['3901681','3902010','3902463','6900790','6901995','6902658','6902766','6902771',
#'6902892','6902910','6902915','6902917','6902930','6902934','6902981','6903136','6903176','6903256',
#'6903260','6904129','6904217','7900497','7900508','7900984']



list_wmos=['6900631','6900632','6900673','6900674','6900675','6900676','6900677','6900678','6900679',
            '6900680','6900681','6900682','6900683','6900684','6900685','6900686','6900939','6900964']

liste_val=['MARTEC','MARTEC','MARTEC','MARTEC','MARTEC','MARTEC','MARTEC','MARTEC','MARTEC',
'MARTEC','MARTEC','MARTEC','MARTEC','MARTEC','NKE','MARTEC','NKE','NKE']

liste_val2=['839','839','839','839','839','839','839','839','839',
'839','839','839','839','839','839','839','839','839']

liste_val3=[6.5018,56.6108,56.3732,56.4918,47.1863,47.2928,48.7112,48.9575]

liste_val4=[-22.9995,-48.0125,-43.9360,-40.9580,-39.0082,-37.3563,-17.0953,-14.2222]

liste_val5=['6339','6340','6341']

In [ ]:
len(list_wmos)

In [ ]:
len(liste_val)

In [ ]:
len(liste_val2)

In [10]:
#list_wmos=['1900067','2900425','6900132','6900162','6900228','6900229','6900236','6900237','6900238','6900239','6900240','6900241','6900242']
#liste_val=['n/a']
#list_wmos=['1900063']

dac="coriolis"
#
wrk_dir="C:/Users/ddobler/Documents/08_DD_scripts/01_Scripts_Metadata_and_Data_Update/Metadata_WorkDir/lot_20251006_prepa/"

#list_wmos=df['PLATFORM_CODE'].values
#list_sensor_mdl=df['SENSOR_MODEL'].values
#list_sensor_fm=df['SENSOR_FIRMWARE_VERSION'].values
#list_prog_name=df['PROGRAM_NAME'].values

for iwmo in range(len(list_wmos)):
#for iwmo in range(10):

    wmo=str(list_wmos[iwmo])
    lsm=list_sensor_mdl[iwmo]
    lsfm=list_sensor_fm[iwmo]
    lpn=list_prog_name[iwmo]
    #nnn_val=liste_val[iwmo]
    #nnn_val2=liste_val2[iwmo]
    #nnn_val3=liste_val3[iwmo]
    #nnn_val4=liste_val4[iwmo]
    #nnn_val5=liste_val5[iwmo]

    history_update_text='updated by D.Dobler for UM 3.44 compliance (Euro-Argo ERIC)'

    # Select wmo to treat
    #wmo="6900203"
    print("\n\n---------------------------")
    print("Treating wmo ",wmo)

  
    # select modifications to perform

    i_new_anomaly_field           = 0
    new_anomaly_field="ASD:AbruptSaltyDrift"

    i_new_program_name            = 1
    new_program_name = lpn
    
    update_deployment_platform_length = 1

    i_new_juld                    = 0
    new_juld = 22256.4784375000

    i_new_depl_ctd                = 0
    new_depl_ctd=""

    i_new_PI_NAME                 = 0
    new_PI_NAME=" "

    i_new_DEPLOYMENT_PLATFORM     = 1
    new_DEPLOYMENT_PLATFORM="Polarstern uri:https://vocab.nerc.ac.uk/collection/C17/current/06AQ/"

    i_new_DEPLOYMENT_CRUISE_ID    = 0
    new_DEPLOYMENT_CRUISE_ID=""

    i_new_launch_longitude        = 0
    new_launch_longitude=""

    i_new_launch_latitude         = 0
    new_launch_latitude=""
    
    i_new_launch_date             = 0
    new_launch_date="20140524185900"
    
    i_new_pf_type                 = 0
    new_pf_type='PROVOR_II'
    
    i_new_pf_maker                = 0
    new_pf_maker=' '

    i_new_wmo_inst_type           = 0
    new_wmo_inst_type=' '

    i_new_trans_system_1          = 0
    new_trans_system_1='IRIDIUM'

    i_new_ptt_1                    = 0
    new_ptt_1 ='2697'
    
    i_new_trans_system_id_1        = 0
    new_trans_system_id_1='n/a'

    i_new_positioning_system_1    = 0
    new_positioning_system_1='GPS'

    
    i_new_float_sn                = 0
    new_float_sn="OIN-13IT-A3-AR-01"
    
    i_new_sensor_type             = 0

    i_new_parameter_sensor_for_psal = 0
    new_parameter_sensor_for_psal="CTD_CNDC"

    i_new_parameter_sensor_for_cndc = 0
    new_parameter_sensor_for_cndc="CTD_CNDC"

    i_new_parameter_sensor_for_temp = 0
    new_parameter_sensor_for_temp="CTD_TEMP"

    i_new_parameter_sensor_for_pres = 0
    new_parameter_sensor_for_pres="CTD_PRES"

    i_new_parameter_sensor_for_down_irr = 0
    new_parameter_sensor_for_down_irr="AUX_RADIOMETER_DOWN_IRR"

    i_new_parameter_sensor_for_up_rad = 0
    new_parameter_sensor_for_up_rad="AUX_RADIOMETER_UP_RAD"

    
    i_new_TS_sensor_maker         = 0
    #new_TS_sensor_maker="SBE"
    new_TS_sensor_maker="FSI"
     
    i_new_TS_model                = 1
    #new_TS_model="SBE41"
    #new_TS_model="FSI"
    new_TS_model=lsm

    i_new_TS_model_firmware       = 1
    new_TS_model_firmware=lsfm
    

    i_new_TS_SN                   = 0
    new_TS_SN="5720"
    
    i_new_P_sensor_maker          = 0
    #new_P_sensor_maker="DRUCK"
    new_P_sensor_maker="KISTLER"
    #new_P_sensor_maker="PAINE"
    #new_P_sensor_maker="FSI"
    
    i_new_P_model                 = 0
    #new_P_model="DRUCK_2900PSIA"
    new_P_model="KISTLER_2900PSIA"
    #new_P_model="PAINE_2900PSIA"
    #new_P_model="FSI"

    i_new_P_model_firmware        = 0
    new_P_model_firmware=""
    
    if wmo in ['1900472','1900473']:
        i_new_TS_sensor_maker         = 1
        new_TS_sensor_maker="SBE"
        i_new_P_sensor_maker          = 1
        new_P_sensor_maker="Unknown"
        i_new_P_model                 = 1
        new_P_model="UNKNOWN"
        if wmo in ['1900472']:
            i_new_TS_SN                   = 1
            new_TS_SN="784"
        if wmo in ['1900473']:
            i_new_TS_SN                   = 1
            new_TS_SN="740"
    if wmo in ['1900386','1900466','1900467','1900468','1900469','1900470','1900471']:
        i_new_TS_sensor_maker         = 1
        new_TS_sensor_maker="Unknown"
        i_new_P_sensor_maker          = 1
        new_P_sensor_maker="Unknown"
        i_new_P_model                 = 1
        new_P_model="UNKNOWN"

    i_new_P_SN                    = 0
    new_P_SN="2147112"
    
    i_new_CHLA_SN                 = 0
    new_CHLA_SN="2663"    

    i_new_DOXY_SN                 = 0
    new_DOXY_SN="0999"    


    
    


    i_history_update_3_1          = 0

    i_ftp_from_ifremer            = 1
    #where to save
    
    if not os.path.exists(wrk_dir):
        os.mkdir(wrk_dir)
    if not os.path.exists(wrk_dir + "/before/"):
        os.mkdir(wrk_dir + "/before/")
    if not os.path.exists(wrk_dir + "/after/"):
        os.mkdir(wrk_dir + "/after/")
    
    file_to_update=wmo + "_meta.nc"
    OUT_FILE=wrk_dir + "/after/" + file_to_update
    IN_FILE = wrk_dir + "/before/" + file_to_update
    print(OUT_FILE)
    
 
    # where to get 
    if i_ftp_from_ifremer:
        print("Getting file from data-argo.ifremer.fr")
        URL = "https://data-argo.ifremer.fr/dac/"+dac+"/"+wmo+"/"+wmo+"_meta.nc"
        #URL = "https://data-argo.ifremer.fr/dac/"+dac+"/"+wmo+"/profiles/"+ file_to_update
        print(URL)
        # commands
        response = requests.get(URL)
        if response.status_code == 404:continue
        open(IN_FILE, "wb").write(response.content)
        shutil.copyfile(IN_FILE,OUT_FILE)
    else:
        print("Getting file from local directory")
        local_FILE="C:/Users/ddobler/Documents/09_Scripts_WD/lot_014_deleted_level_in/" + wmo + "_meta.nc"
        shutil.copyfile(local_FILE,OUT_FILE)

    #print("Making changes for wmo ",wmo)

    if update_deployment_platform_length:
        print("changing DEPLOYMENT_PLATFORM length to STRING128")
        ds_src=Dataset(IN_FILE,'r')
        ds=change_variable_dimension_length_1D(OUT_FILE,ds_src,'DEPLOYMENT_PLATFORM',128)
    else:
        ds=Dataset(OUT_FILE,'a')

    ## add new fields if necessary
    for varname in ['PROGRAM_NAME','SENSOR_FIRMWARE_VERSION']:
        try:
            var=ds.variables[varname]
        except:
            print("creating the new variable " + varname)
            add_new_var(ds,varname)
    change_var_long_name(ds,'PROJECT_NAME','Name of the project')
    

    # update the global attribute history:
    date_courante=dt.datetime.now(dt.timezone.utc)
    date_courante_str=date_courante.strftime("%Y-%m-%dT%H:%M:%SZ")
    #lhistory=date_courante_str+ ' creation; '+date_courante_str+' last update (coriolis float real time data processing)'


    # First define the level corresponding to the wanted sensor
    SENSOR_TYPE=ds.variables["SENSOR"]
    SENSOR_MAKER=ds.variables["SENSOR_MAKER"]
    SENSOR_MODEL=ds.variables["SENSOR_MODEL"]
    SENSOR_SN=ds.variables["SENSOR_SERIAL_NO"]
    FLOAT_SN=ds.variables["FLOAT_SERIAL_NO"]
    #
    #print(Sensor_type.shape)
    for i_sensor in range(SENSOR_TYPE.shape[0]):
        sensor_str=SENSOR_TYPE[i_sensor,:].tobytes().decode().strip()
    #    
        if i_new_TS_SN | i_new_P_SN | i_new_CHLA_SN | i_new_DOXY_SN | i_new_TS_model | i_new_P_model | \
        i_new_sensor_type | i_new_TS_sensor_maker | i_new_P_sensor_maker:
            print("i_sensor=",i_sensor,"sensor_str=",sensor_str)
    #        
        if (sensor_str == "CTD_TEMP") | (sensor_str == "TEMP"):
            i_T=i_sensor
        if (sensor_str == "CTD_CNDC") | (sensor_str == "CNDC"):
            i_S=i_sensor
        if (sensor_str == "CTD_PRES") | (sensor_str == "PRES"):
            i_P=i_sensor
        if (sensor_str == "FLUOROMETER_CHLA"):
            i_Chla=i_sensor
        if (sensor_str == "OPTODE_DOXY"):
            i_Doxy=i_sensor

    if i_new_TS_SN:
        change_val_2D_char(ds,"SENSOR_SERIAL_NO",i_T,new_TS_SN)
        change_val_2D_char(ds,"SENSOR_SERIAL_NO",i_S,new_TS_SN)

    if i_new_P_SN:    
        change_val_2D_char(ds,"SENSOR_SERIAL_NO",i_P,new_P_SN)
        
    if i_new_CHLA_SN:
        change_val_2D_char(ds,"SENSOR_SERIAL_NO",i_Chla,new_CHLA_SN)
        
    if i_new_DOXY_SN:     
        change_val_2D_char(ds,"SENSOR_SERIAL_NO",i_Doxy,new_DOXY_SN)

    if i_new_TS_model:
        change_val_2D_char(ds,"SENSOR_MODEL",i_T,new_TS_model)
        change_val_2D_char(ds,"SENSOR_MODEL",i_S,new_TS_model)

    if i_new_TS_model_firmware:
        change_val_2D_char(ds,"SENSOR_FIRMWARE_VERSION",i_T,new_TS_model_firmware)
        change_val_2D_char(ds,"SENSOR_FIRMWARE_VERSION",i_S,new_TS_model_firmware)
        
    if i_new_P_model:
        change_val_2D_char(ds,"SENSOR_MODEL",i_P,new_P_model)

    if i_new_P_model_firmware:
        change_val_2D_char(ds,"SENSOR_FIRMWARE_VERSION",i_P,new_P_model_firmware)

    if i_new_sensor_type:
        change_val_2D_char(ds,"SENSOR_TYPE",i_P,"CTD_PRES")
        change_val_2D_char(ds,"SENSOR_TYPE",i_S,"CTD_CNDC")
        change_val_2D_char(ds,"SENSOR_TYPE",i_T,"CTD_TEMP")
        

    if i_new_TS_sensor_maker:
        change_val_2D_char(ds,"SENSOR_MAKER",i_T,new_TS_sensor_maker)
        change_val_2D_char(ds,"SENSOR_MAKER",i_S,new_TS_sensor_maker)

    if i_new_P_sensor_maker:
        change_val_2D_char(ds,"SENSOR_MAKER",i_P,new_P_sensor_maker)


    if i_new_trans_system_1:
        change_val_2D_char(ds,"TRANS_SYSTEM",0,new_trans_system_1)


    if i_new_trans_system_id_1:
        change_val_2D_char(ds,"TRANS_SYSTEM_ID",0,new_trans_system_id_1)

    if i_new_anomaly_field:
        change_val_1D_char(ds,"ANOMALY",new_anomaly_field)

    if i_new_ptt_1:
        change_val_1D_char(ds,"PTT",new_ptt_1)

    if i_new_depl_ctd:
        change_val_1D_char(ds,"DEPLOYMENT_REFERENCE_STATION_ID",new_depl_ctd)

    if i_new_positioning_system_1:
        change_val_2D_char(ds,"POSITIONING_SYSTEM",0,new_positioning_system_1)

    PARAMETER=ds.variables["PARAMETER"]
    for i_parameter in range(PARAMETER.shape[0]):
        
        parameter_str=PARAMETER[i_parameter,:].tobytes().decode().strip()
        if i_new_parameter_sensor_for_psal | i_new_parameter_sensor_for_cndc | i_new_parameter_sensor_for_temp | i_new_parameter_sensor_for_pres | \
           i_new_parameter_sensor_for_down_irr | i_new_parameter_sensor_for_up_rad :
            print("i_parameter=",i_parameter,"sensor_str=",parameter_str)
        if (parameter_str == "TEMP"): i_p_T=i_parameter
        if (parameter_str == "CNDC"): i_p_C=i_parameter
        if (parameter_str == "PRES"): i_p_P=i_parameter
        if (parameter_str == "PSAL"): i_p_S=i_parameter
        if (parameter_str == 'DOWN_IRRADIANCE'): i_p_di=i_parameter
        if (parameter_str == 'UP_RADIANCE'): i_p_ur=i_parameter
            

            
    if i_new_parameter_sensor_for_psal:
        change_val_2D_char(ds,"PARAMETER_SENSOR",i_p_S,new_parameter_sensor_for_psal)

    if i_new_parameter_sensor_for_cndc:
        change_val_2D_char(ds,"PARAMETER_SENSOR",i_p_C,new_parameter_sensor_for_cndc)

    if i_new_parameter_sensor_for_temp:
        change_val_2D_char(ds,"PARAMETER_SENSOR",i_p_T,new_parameter_sensor_for_temp)
        
    if i_new_parameter_sensor_for_pres:
        change_val_2D_char(ds,"PARAMETER_SENSOR",i_p_P,new_parameter_sensor_for_pres)
        
    if i_new_parameter_sensor_for_down_irr:
        change_val_2D_char(ds,"PARAMETER_SENSOR",i_p_di,new_parameter_sensor_for_down_irr)
        
    if i_new_parameter_sensor_for_up_rad:
        change_val_2D_char(ds,"PARAMETER_SENSOR",i_p_ur,new_parameter_sensor_for_up_rad)

    if i_history_update_3_1:
        print("Modifying global attribute section")
        ds.title = "Argo float metadata file"
        ds.institution = "CORIOLIS"
        ds.source = "Argo float"
        ds.history = lhistory
        ds.references = "http://www.argodatamgt.org/Documentation" ;
        # ds.user_manual_version = "--" ;
        # ds.Conventions = "--" ;


    if i_new_launch_longitude:
        l_lon=ds.variables["LAUNCH_LONGITUDE"]
        print("Modifying launch longitude from ", \
              l_lon[:]," to ",new_launch_longitude)
        l_lon[:]=new_launch_longitude
        
    if i_new_launch_latitude:
        l_lat=ds.variables["LAUNCH_LATITUDE"]
        print("Modifying launch latitude from ", \
              l_lat[:]," to ",new_launch_latitude)
        l_lat[:]=new_launch_latitude

    if i_new_juld:
        l_juld=ds.variables["JULD"]
        print("Modifying JULD from ", l_juld[:]," to ",new_juld)
        l_juld[:]=new_juld
        l_juld_loc=ds.variables["JULD_LOCATION"]
        print("Modifying JULD_LOCATION from ", l_juld_loc[:]," to ",new_juld)
        l_juld_loc[:]=new_juld
        
    if i_new_launch_date:
        change_val_1D_char(ds,"LAUNCH_DATE",new_launch_date)
        change_val_1D_char(ds,"START_DATE",new_launch_date)
        
    if i_new_float_sn: change_val_1D_char(ds,"FLOAT_SERIAL_NO",new_float_sn)
                
    if i_new_pf_type:
        try:
            varname="PLATFORM_TYPE"
            var=ds.variables[varname]
        except:
            varname="PLATFORM_MODEL"
            var=ds.variables[varname]
        change_val_1D_char(ds,varname,new_pf_type)     
        
    if i_new_pf_maker: change_val_1D_char(ds,"PLATFORM_MAKER",new_pf_maker)

    if i_new_wmo_inst_type: change_val_1D_char(ds,"WMO_INST_TYPE",new_wmo_inst_type)

    if i_new_PI_NAME: change_val_1D_char(ds,"PI_NAME",new_PI_NAME)
        
    if i_new_program_name: change_val_1D_char(ds,"PROGRAM_NAME",new_program_name)

    if i_new_DEPLOYMENT_PLATFORM: change_val_1D_char(ds,"DEPLOYMENT_PLATFORM",new_DEPLOYMENT_PLATFORM)
        
    if i_new_DEPLOYMENT_CRUISE_ID: change_val_1D_char(ds,"DEPLOYMENT_CRUISE_ID",new_DEPLOYMENT_CRUISE_ID)

    print("\n\n Checks section")
    if i_new_P_model|i_new_TS_model|i_new_P_SN|i_new_TS_SN|i_new_sensor_type:
        print("i_T=",i_T," i_S=",i_S,"i_P=",i_P)
    if i_new_sensor_type:
        print("\n New SENSOR_TYPE")
        print(SENSOR_TYPE[:,:].tobytes().decode().strip())
    if i_new_P_model|i_new_TS_model: 
        print("\n New SENSOR_MAKER")
        print(SENSOR_MAKER[:,:].tobytes().decode().strip())
        print("\n New SENSOR_MODEL")
        print(SENSOR_MODEL[:,:].tobytes().decode().strip())
    if i_new_P_SN|i_new_TS_SN:
        print("\n New SENSOR_SN")
        print(SENSOR_SN[:,:].tobytes().decode().strip())
    if i_new_float_sn:
        print("\n New FLOAT_SERIAL_NO")
        print(FLOAT_SN[:].tobytes().decode().strip())
    if i_new_parameter_sensor_for_psal:
        print("\n New PARAMETER_SENSOR")
        print(PARAMETER_SENSOR[:].tobytes().decode().strip())

    #print("check")
    #print(ds.history)
    
    print("\n Updating history global attribute")

    tmp=str(ds.variables["DATE_CREATION"][:].tobytes(),'utf-8')
    record_date_creation=tmp[:4]+"-"+tmp[4:6]+"-"+tmp[6:8]+"T"+tmp[8:10]+":"+tmp[10:12]+":"+tmp[12:14]+"Z"
    #print(record_date_creation)


    # update the global attribute history:
    date_courante=dt.datetime.now(dt.timezone.utc)
    date_courante_str=date_courante.strftime("%Y-%m-%dT%H:%M:%SZ")
    lhistory=record_date_creation+ ' creation; '+date_courante_str + ' ' + history_update_text 
    #lhistory=record_date_creation+ ' creation; '+date_courante_str+' updated'
    try:
        print("Old history global attr: ",ds.history)
    except:
        print("Old history global attr did not exist")
    ds.history = lhistory
    print("New history global attr: ",ds.history)

    print("Updating DATE_UPDATE")
    varname="DATE_UPDATE"
    new_val=date_courante.strftime("%Y%m%d%H%M%S")
    change_val_1D_char(ds,varname,new_val)


    
    
#     # Special treatment for 6900952 : change dimension for PREDEPLOYMENT_CALIB_* parameter to indicate a char 4096 dimension
    
#     # Create a new NetCDF file
#     new_ds = Dataset(OUT_FILE_2, 'w')

#     new_ds.createDimension("STRING4096",4096)
    
#     # Copy all dimensions and variables except the one you want to delete
#     for dimname, dim in ds.dimensions.items():
#         if dimname == 'N_MISSIONS':
#             new_ds.createDimension(dimname, None)
#         else:
#             new_ds.createDimension(dimname, len(dim))
        

#     for varname, var in ds.variables.items():
#         if varname not in ['PREDEPLOYMENT_CALIB_COEFFICIENT',"PREDEPLOYMENT_CALIB_COMMENT","PREDEPLOYMENT_CALIB_EQUATION"]:
#             new_var = new_ds.createVariable(varname, var.dtype, var.dimensions,fill_value=ds.variables[varname]._FillValue)
#             new_var[:] = var[:]
#             new_var.long_name = var.long_name

#             for local_attrs in ds.variables[varname].ncattrs():
#                 if (local_attrs != '_FillValue'):
#                     new_var.setncattr(local_attrs,ds.variables[varname].getncattr(local_attrs))
            
#     new_ds.createVariable("PREDEPLOYMENT_CALIB_EQUATION","S1",("N_PARAM","STRING4096"),fill_value=" ")
#     new_ds.createVariable("PREDEPLOYMENT_CALIB_COEFFICIENT","S1",("N_PARAM","STRING4096"),fill_value=" ")
#     new_ds.createVariable("PREDEPLOYMENT_CALIB_COMMENT","S1",("N_PARAM","STRING4096"),fill_value=" ")
    
#     C_EQ_2=new_ds.variables["PREDEPLOYMENT_CALIB_EQUATION"]
#     C_COEF_2=new_ds.variables["PREDEPLOYMENT_CALIB_COEFFICIENT"]
#     C_CMT_2=new_ds.variables["PREDEPLOYMENT_CALIB_COMMENT"]
    
#     C_EQ=ds.variables["PREDEPLOYMENT_CALIB_EQUATION"]
#     C_COEF=ds.variables["PREDEPLOYMENT_CALIB_COEFFICIENT"]
#     C_CMT=ds.variables["PREDEPLOYMENT_CALIB_COMMENT"]
    
#     C_EQ_2.long_name=C_EQ.long_name
#     C_COEF_2.long_name=C_COEF.long_name
#     C_CMT_2.long_name=C_CMT.long_name
    
#     texte="n/a"
#     new_array=np.chararray(C_EQ_2.shape)
#     new_array[:]=" "
#     for i in range(len(texte)):
#         new_array[:,i]=texte[i]
#     C_EQ_2[:]=new_array[:]
#     C_COEF_2[:]=new_array[:]
    
#     global_attributes_list=ds.ncattrs()
#     for attr in global_attributes_list:
#         new_ds.setncattr(attr,ds.getncattr(attr))

#     new_ds.close()
    

    ds.close()



---------------------------
Treating wmo  7900516
C:/Users/ddobler/Documents/08_DD_scripts/01_Scripts_Metadata_and_Data_Update/Metadata_WorkDir/lot_20251006_prepa//after/7900516_meta.nc
Getting file from data-argo.ifremer.fr
https://data-argo.ifremer.fr/dac/coriolis/7900516/7900516_meta.nc
changing DEPLOYMENT_PLATFORM length to STRING128
creating the new variable PROGRAM_NAME
creating the new variable SENSOR_FIRMWARE_VERSION
i_sensor= 0 sensor_str= FLOATCLOCK_MTIME
i_sensor= 1 sensor_str= CTD_PRES
i_sensor= 2 sensor_str= CTD_TEMP
i_sensor= 3 sensor_str= CTD_CNDC
Modifying  SENSOR_MODEL  for dim_level  2 from  SBE41CP_V7.2.5  to  SBE41CP
Modifying  SENSOR_MODEL  for dim_level  3 from  SBE41CP_V7.2.5  to  SBE41CP
Modifying  SENSOR_FIRMWARE_VERSION  for dim_level  2 from    to  7.2.5
Modifying  SENSOR_FIRMWARE_VERSION  for dim_level  3 from    to  7.2.5
Modifying  PROGRAM_NAME  from    to  Argo BSH
Modifying  DEPLOYMENT_PLATFORM  from  POLARSTERN  to  Polarstern uri:https://vocab.nerc.a

In [ ]:
print("wmo=",wmo)

# check step
ds=Dataset(OUT_FILE,'r')
SENSOR_TYPE=ds.variables["SENSOR"]
SENSOR_MAKER=ds.variables["SENSOR_MAKER"]
SENSOR_MODEL=ds.variables["SENSOR_MODEL"]
SENSOR_SN=ds.variables["SENSOR_SERIAL_NO"]
DATE_UPDATE=ds.variables["DATE_UPDATE"]
LAUNCH_LATITUDE=ds.variables["LAUNCH_LATITUDE"]
LAUNCH_LONGITUDE=ds.variables["LAUNCH_LONGITUDE"]
LAUNCH_DATE=ds.variables["LAUNCH_DATE"]
START_DATE=ds.variables["START_DATE"]
PLATFORM_MAKER=ds.variables["PLATFORM_MAKER"]

hh=ds.history

print("\n--global attribute history")
print(hh)

print("\n--PLATFORM_MAKER:")
print(PLATFORM_MAKER[:].tobytes().decode().strip())

print("\n--SENSOR_TYPE:")
print(SENSOR_TYPE[:,:].tobytes().decode().strip())

print("\n--SENSOR_MAKER:")
print(SENSOR_MAKER[:,:].tobytes().decode().strip())

print("\n--SENSOR_MODEL:")
print(SENSOR_MODEL[:,:].tobytes().decode().strip())

print("\n--SENSOR_SN:")
print(SENSOR_SN[:,:].tobytes().decode().strip())

print("\n--LAUNCH_LONGITUDE:")
print(LAUNCH_LONGITUDE[:])

print("\n--LAUNCH_LATITUDE:")
print(LAUNCH_LATITUDE[:])

print("\n--LAUNCH_DATE:")
print(LAUNCH_DATE[:].tobytes().decode().strip())

print("\n--START_DATE:")
print(START_DATE[:].tobytes().decode().strip())

print("\n--DATE_UPDATE:")
print(DATE_UPDATE[:].tobytes().decode().strip())

ds.close()